# Examen ETL - Notebook Paso a Paso

Este notebook documenta todo el proyecto realizado: generación de datos sintéticos con errores, profiling, limpieza, merge, CDC y Business Intelligence.


## 0. Configuración

Primero definimos rutas base y una pequeña utilidad para mostrar archivos generados durante el flujo.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display, Image, Markdown

BASE_DIR = Path.cwd()
BASE_DIR


In [ ]:
def run_script(script_name: str):
    result = subprocess.run(
        ['python3', script_name],
        cwd=BASE_DIR,
        capture_output=True,
        text=True,
        check=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)


## 1. Generación de datasets sucios

Se generan dos datasets sintéticos con errores intencionales:

- `Employee Information`: valores faltantes, fechas inconsistentes, género inconsistente, salarios inválidos.
- `Sales Data`: categorías faltantes, cantidades negativas y fechas inconsistentes.

In [ ]:
run_script('generate_dirty_datasets.py')

In [ ]:
df_employee_dirty = pd.read_csv(BASE_DIR / 'employee_information_dirty.csv')
df_sales_dirty = pd.read_csv(BASE_DIR / 'sales_data_dirty.csv')

print('Employee dirty shape:', df_employee_dirty.shape)
print('Sales dirty shape:', df_sales_dirty.shape)

display(df_employee_dirty.head())
display(df_sales_dirty.head())


In [ ]:
employee_missing_pct = round(df_employee_dirty.isna().sum().sum() / df_employee_dirty.size * 100, 2)
sales_category_missing_pct = round(df_sales_dirty['Product Category'].isna().mean() * 100, 2)
sales_negative_qty = int((pd.to_numeric(df_sales_dirty['Quantity Sold'], errors='coerce') < 0).sum())

print('Employee missing %:', employee_missing_pct)
print('Sales Product Category missing %:', sales_category_missing_pct)
print('Sales negative quantity rows:', sales_negative_qty)


## 2. Data Profiling

En esta etapa se inspecciona la estructura, tipos, estadísticas, missing values, duplicados y errores de formato.
Además, se generan heatmaps para visualizar la distribución de valores faltantes.

In [ ]:
run_script('profile_datasets.py')

In [ ]:
profiling_report = (BASE_DIR / 'data_profiling_report.txt').read_text(encoding='utf-8')
print(profiling_report)


In [ ]:
display(Image(filename=str(BASE_DIR / 'employee_missing_heatmap.png')))
display(Image(filename=str(BASE_DIR / 'sales_missing_heatmap.png')))


## 3. Data Cleaning

Se corrigen y estandarizan los datos con las siguientes técnicas:

- `KNNImputer` para variables numéricas.
- `Most Frequent Imputer` para variables categóricas.
- Parsing e imputación de fechas.
- Estandarización de texto y categorías.

In [ ]:
run_script('clean_datasets.py')

In [ ]:
df_employee_clean = pd.read_csv(BASE_DIR / 'employee_information_cleaned.csv')
df_sales_clean = pd.read_csv(BASE_DIR / 'sales_data_cleaned.csv')

print('Employee cleaned missing cells:', int(df_employee_clean.isna().sum().sum()))
print('Sales cleaned missing cells:', int(df_sales_clean.isna().sum().sum()))

display(df_employee_clean.head())
display(df_sales_clean.head())


In [ ]:
cleaning_report = (BASE_DIR / 'data_cleaning_report.txt').read_text(encoding='utf-8')
print(cleaning_report)


## 4. Merging Data

Después de limpiar los datasets, se fusionan para dejar un dataset listo para análisis de negocio.

Nota: como el dataset de ventas original no incluía `Employee ID`, en el flujo se añadió una asignación determinística antes del merge.

In [ ]:
run_script('merge_datasets.py')

In [ ]:
df_merged = pd.read_csv(BASE_DIR / 'employee_sales_merged.csv')
print('Merged shape:', df_merged.shape)
print('Merged nulls:', int(df_merged.isna().sum().sum()))
display(df_merged.head())


In [ ]:
merge_report = (BASE_DIR / 'data_merge_report.txt').read_text(encoding='utf-8')
print(merge_report)


## 5. Change Data Capture (CDC)

Se detectan registros nuevos, eliminados y modificados entre la versión original y la versión limpia de cada dataset.

In [ ]:
run_script('cdc_tracking.py')

In [ ]:
df_employee_cdc = pd.read_csv(BASE_DIR / 'employee_cdc_changes.csv')
df_sales_cdc = pd.read_csv(BASE_DIR / 'sales_cdc_changes.csv')

print('Employee CDC rows:', len(df_employee_cdc))
print('Sales CDC rows:', len(df_sales_cdc))
display(df_employee_cdc.head(10))
display(df_sales_cdc.head(10))


In [ ]:
cdc_report = (BASE_DIR / 'cdc_report.txt').read_text(encoding='utf-8')
print(cdc_report)


## 6. Business Intelligence Analysis

Se generan visualizaciones y métricas sobre el dataset merged para extraer insights de negocio.

In [ ]:
run_script('business_intelligence_analysis.py')

In [ ]:
bi_report = (BASE_DIR / 'business_intelligence_report.txt').read_text(encoding='utf-8')
print(bi_report)


In [ ]:
display(Image(filename=str(BASE_DIR / 'bi_sales_by_department.png')))
display(Image(filename=str(BASE_DIR / 'bi_sales_by_product_category.png')))
display(Image(filename=str(BASE_DIR / 'bi_salary_vs_performance.png')))
display(Image(filename=str(BASE_DIR / 'bi_payment_method_vs_sale_amount.png')))


## 7. Dashboard Interactiva

Además del notebook, se construyó una app de Streamlit para explorar todo el proyecto de forma interactiva.

In [ ]:
print('Dashboard app:', BASE_DIR / 'dashboard_app.py')
print('Para ejecutarla localmente:')
print('python3 -m streamlit run dashboard_app.py')


## 8. Conclusiones

Con este notebook queda documentado el flujo completo ETL/BI:

1. Generación de datos sintéticos con errores.
2. Profiling para entender calidad y estructura.
3. Limpieza con imputación y estandarización.
4. Integración de datasets.
5. Trazabilidad de cambios con CDC.
6. Visualización e interpretación de resultados de negocio.